# App Lifecycle Manager

Monitors a Databricks App for user activity and stops it when idle.
Also supports starting the app on-demand (e.g., scheduled for business hours).

In [0]:
dbutils.widgets.text("app_name", "genie-curator-workbench", "App Name")
dbutils.widgets.dropdown("action", "check_idle", ["check_idle", "start", "stop"], "Action")
dbutils.widgets.text("idle_minutes", "60", "Idle Minutes (for check_idle)")

APP_NAME = dbutils.widgets.get("app_name")
ACTION = dbutils.widgets.get("action")
IDLE_MINUTES = int(dbutils.widgets.get("idle_minutes"))

print(f"App: {APP_NAME} | Action: {ACTION} | Idle threshold: {IDLE_MINUTES}m")

In [0]:
import time, requests, json
from datetime import datetime, timezone, timedelta
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
HOST = w.config.host.rstrip("/")
_WH_ID = "e3a0fc2c08db05bb"      # shared SQL warehouse
_HB_CATALOG = "gap_demo_dev_app_monitoring"  # shared monitoring catalog (user-agnostic)

def get_app_info(name):
    """Get app status and metadata via REST API."""
    headers = w.config.authenticate()
    r = requests.get(f"{HOST}/api/2.0/apps/{name}", headers=headers, timeout=15)
    r.raise_for_status()
    d = r.json()
    compute_state = d.get("compute_status", {}).get("state", "UNKNOWN")
    update_time = d.get("update_time", "")
    url = d.get("url", "")
    return {"compute_state": compute_state, "update_time": update_time, "url": url, "raw": d}

def _run_sql(sql):
    """Run SQL via Statement Execution API; return rows list."""
    h = dict(w.config.authenticate())
    h["Content-Type"] = "application/json"
    r = requests.post(f"{HOST}/api/2.0/sql/statements", headers=h, json={
        "warehouse_id": _WH_ID, "statement": sql,
        "wait_timeout": "30s", "disposition": "INLINE",
    }, timeout=35)
    r.raise_for_status()
    body = r.json()
    status = body.get("status", {}).get("state", "")
    if status != "SUCCEEDED":
        raise RuntimeError(f"SQL {status}: {body.get('status',{}).get('error',{}).get('message','')}")
    chunk = body.get("result", {}).get("data_array", [])
    cols = [c["name"] for c in body.get("manifest", {}).get("schema", {}).get("columns", [])]
    return [dict(zip(cols, row)) for row in chunk]

def get_recent_activity(app_name, minutes=60):
    """Check for recent user activity via the Delta heartbeat table.
    The Dash app persists {app, ts, user} every 5 min via SQL Statement API.
    Falls back to update_time from the Apps API if the table is unavailable.
    Returns (has_activity: bool, last_ts: str | None, method: str)."""
    now = datetime.now(timezone.utc)
    cutoff = now - timedelta(minutes=minutes)

    # Method 1: Read the heartbeat Delta table (app writes to base catalog)
    try:
        rows = _run_sql(
            f"SELECT ts, user FROM {_HB_CATALOG}.genie_workbench.app_heartbeat "
            f"WHERE app = '{app_name}' ORDER BY ts DESC LIMIT 1"
        )
        if rows:
            ts_str = rows[0]["ts"]
            user = rows[0]["user"]
            last_ts = datetime.fromisoformat(str(ts_str).replace("Z", "+00:00").replace(" ", "T"))
            # If tz-naive, assume UTC
            if last_ts.tzinfo is None:
                last_ts = last_ts.replace(tzinfo=timezone.utc)
            print(f"  Heartbeat table: ts={ts_str}, user={user}")
            if last_ts > cutoff:
                return True, ts_str, "heartbeat table"
            else:
                return False, ts_str, "heartbeat table"
    except Exception as e:
        print(f"  Heartbeat table unavailable: {e}")

    # Method 2: Fall back to update_time from app API (tracks config changes only)
    try:
        info = get_app_info(app_name)
        ut = info.get("update_time", "")
        if ut:
            update_dt = datetime.fromisoformat(ut.replace("Z", "+00:00"))
            if update_dt > cutoff:
                return True, ut, "update_time (weak signal)"
            return False, ut, "update_time (weak signal)"
    except Exception as e:
        print(f"  App info fallback failed: {e}")

    # No data — do NOT stop (be safe)
    print("  WARNING: No activity data available. NOT stopping to be safe.")
    return True, None, "no data (conservative)"

# Ensure the shared monitoring catalog + schema exist
try:
    _run_sql(f"CREATE CATALOG IF NOT EXISTS {_HB_CATALOG}")
    # Shared monitoring catalog: owned by account users so ALL users can access
    _run_sql(f"ALTER CATALOG {_HB_CATALOG} SET OWNER TO `account users`")
    _run_sql(f"CREATE SCHEMA IF NOT EXISTS {_HB_CATALOG}.genie_workbench")
    _run_sql(f"ALTER SCHEMA {_HB_CATALOG}.genie_workbench SET OWNER TO `account users`")
    _run_sql(f"CREATE TABLE IF NOT EXISTS {_HB_CATALOG}.genie_workbench.app_heartbeat "
             f"(app STRING, ts TIMESTAMP, user STRING) USING DELTA")
    _run_sql(f"ALTER TABLE {_HB_CATALOG}.genie_workbench.app_heartbeat SET OWNER TO `account users`")
    print(f"Monitoring catalog ready: {_HB_CATALOG}.genie_workbench.app_heartbeat (owned by account users)")
except Exception as e:
    print(f"WARNING: Could not bootstrap monitoring table: {e}")

print("SDK ready.")

In [0]:
info = get_app_info(APP_NAME)
state = info["compute_state"]
print(f"Current state: {state}")
print(f"URL: {info['url']}")
print(f"Last update: {info['update_time']}")

headers_rest = w.config.authenticate()

def _stop_app(name):
    """Stop app via REST API with polling."""
    r = requests.post(f"{HOST}/api/2.0/apps/{name}/stop", headers=headers_rest, timeout=15)
    if r.status_code not in (200, 201):
        print(f"Stop API returned {r.status_code}: {r.text[:200]}")
        return False
    for _ in range(24):
        time.sleep(5)
        check = get_app_info(name)
        if check["compute_state"] == "STOPPED":
            print(f"App stopped successfully.")
            return True
    print(f"App stop initiated but still transitioning.")
    return False

def _start_app(name):
    """Start app via REST API with polling."""
    r = requests.post(f"{HOST}/api/2.0/apps/{name}/start", headers=headers_rest, timeout=15)
    if r.status_code not in (200, 201):
        print(f"Start API returned {r.status_code}: {r.text[:200]}")
        return False
    for _ in range(60):
        time.sleep(5)
        check = get_app_info(name)
        if check["compute_state"] == "ACTIVE":
            print(f"App started. URL: {check['url']}")
            return True
    print(f"App start initiated but still transitioning.")
    return False

if ACTION == "check_idle":
    if state != "ACTIVE":
        print(f"App is already {state} — nothing to do.")
    else:
        has_activity, last_ts, method = get_recent_activity(APP_NAME, IDLE_MINUTES)
        print(f"Activity check: active={has_activity} | last={last_ts} | method={method}")
        if not has_activity:
            print(f"No activity in the last {IDLE_MINUTES} minutes — stopping app...")
            _stop_app(APP_NAME)
        else:
            print(f"App has recent activity — keeping it running.")

elif ACTION == "start":
    if state == "ACTIVE":
        print(f"App is already ACTIVE — nothing to do.")
    else:
        print(f"Starting app {APP_NAME}...")
        _start_app(APP_NAME)

elif ACTION == "stop":
    if state == "STOPPED":
        print(f"App is already STOPPED — nothing to do.")
    else:
        print(f"Stopping app {APP_NAME}...")
        _stop_app(APP_NAME)

print("\nDone.")